## What This Notebook Does

This notebook trains machine learning models to predict Formula 1 race outcomes
using the 19 engineered features from Notebook 03.

We train models for **three target variables:**

| Target | Problem Type | Description |
|--------|-------------|-------------|
| `position` | Regression | Predict the driver's exact finishing position (1–20) |
| `points_finish` | Classification | Predict whether the driver scores points (0/1) |
| `points` | Regression | Predict the exact championship points scored |

We follow a two-stage approach:

**Stage 1 — Baseline Models** (simple, interpretable):
- Linear Regression (for position and points)
- Logistic Regression (for points_finish)

**Stage 2 — Advanced Models** (powerful, non-linear):
- Random Forest
- Gradient Boosting
- XGBoost

The best model from each stage is selected by performance on the
**validation set** and saved for final evaluation in Notebook 06.



## 🧠 Key Modelling Decisions

### Why Three Target Variables?

Each target answers a different, useful question:

- **`position`**: "Where will this driver finish?" — most granular prediction.
  Used for full race order prediction and the custom reward score.

- **`points_finish`**: "Will this driver score points?" — most actionable prediction.
  Scoring points (top 10) is the line between a good race and a bad one.

- **`points`**: "How many points will this driver score?" — bridges the two above.
  Accounts for the non-linear points system (P1=25, P2=18, P3=15...).

---

### Why NOT a Random Train-Test Split?

In standard ML we randomly shuffle data into train/test.
For time-series data like F1 race results, this causes leakage:

- Random split: train data contains 2025 races, test data contains 2023 races.
- The model learns 2025 driver form to predict 2023 results — invalid.

**We use a temporal (season-based) split:**
- **Train:** seasons 2018–2023 (6 seasons of racing history)
- **Validation:** season 2024 (used to tune and compare models)
- **Test:** seasons 2025–2026 (held out — only used in Notebook 06)

This guarantees the model only ever predicts the future using the past.

---

### Pipeline Architecture

For each target variable:
```
Raw features → Preprocessing → Model → Predictions → Metrics
```

Linear models require **feature scaling** (StandardScaler).
Tree-based models do NOT require scaling (they use splits, not distances).
We handle this cleanly with sklearn Pipelines.

# Import libs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

#sklearn - preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


#sklearn - baseline models
from sklearn.linear_model import LinearRegression, LogisticRegression

#sklearn - advanced models
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

#sklearn - metrcs
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

#XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print('XGBoost available:', xgb.__version__)
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed. Run: pip install xgboost')

#display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

#output dir
os.makedirs('../output/models', exist_ok=True)

print('All imports complete')
print('Models will be saved to: ../output/models/')


# Load Data

In [ ]:
df = pd.read_csv('../data/processed/f1_features_2018_2026.csv')

# Define Features & Targets

In [ ]:
FEATURES = [
    
    # Base feature
    'grid',
    # Driver Form
    'driver_avg_finish_last_3_races',
    'driver_avg_points_last_3_races',
    'driver_avg_grid_last_3_races',
    'driver_points_cur_season',
    'driver_wins_cur_season',
    'driver_podiums_cur_season',
    'driver_dnf_rate',
    # Constructor Form
    'constructor_avg_finish_last_3_races',
    'constructor_avg_points_last_3_races',
    'constructor_points_current_season',
    'constructor_wins_current_season',
    'constructor_reliability',
    # Track Performance
    'driver_avg_finish_on_circuit',
    'driver_avg_points_on_circuit',
    'driver_best_finish_on_circuit',
    'driver_dnf_on_circuit',
    'driver_track_strength_score',
    # Teammate Comparison
    'driver_beats_teammate_rate',
    'driver_vs_teammate_finish_gap'
]

TARGETS = {
    'position': 'Finish Position (Regression)',
    'points_finish': 'Points Finis (Classification)',
    'points': 'Champoinship Points (Regression)'
}

f'Feature set: {len(FEATURES)} feature'

In [ ]:
for i, f in enumerate(FEATURES, 1):
    print(f'{i:2d}. {f}')

In [ ]:
print(f'Target variables: {len(TARGETS)}')
for col, desc in TARGETS.items():
    print(f'{col:<15} -> {desc}')

In [ ]:
missing = [f for f in FEATURES if f not in df.columns]
missing_targets = [t for t in TARGETS if t not in df.columns]

print(f"Missing features: {missing if missing else "None ✅"}")
print(f"Missing targets: {missing_targets if missing_targets else "None ✅"}")

# Train / Valid / Test

In [ ]:
TRAIN_SEASONS = list(range(2018, 2024))
VAL_SEASONS = [2024]
TEST_SEASONS = [2025, 2026]

train_mask = df.season.isin(TRAIN_SEASONS)
val_mask = df.season.isin(VAL_SEASONS)
test_mask = df.season.isin(TEST_SEASONS)

X_train = df[train_mask][FEATURES].values
X_val = df[val_mask][FEATURES].values
X_test = df[test_mask][FEATURES].values

y_train = {
    'position': df[train_mask]['position'].values,
    'points_finish': df[train_mask]['points_finish'].values,
    'points': df[train_mask]['points'].values,
}
y_val = {
    'position': df[val_mask]['position'].values,
    'points_finish': df[val_mask]['points_finish'].values,
    'points': df[val_mask]['points'].values,
}
y_test = {
    'position': df[test_mask]['position'].values,
    'points_finish': df[test_mask]['points_finish'].values,
    'points': df[test_mask]['points'].values,
}

In [ ]:
print('Temporal split complete:')
print(f'Train (2018-2023): {X_train.shape[0]:>5,} rows'
      f'({X_train.shape[0]/len(df)*100:.1f}% of data)')

print(f'Validate (2024): {X_val.shape[0]:>5,} rows'
      f'({X_val.shape[0]/len(df)*100:.1f}% of data)')


print(f'Test (2025-2026): {X_test.shape[0]:>5,} rows'
      f'({X_test.shape[0]/len(df)*100:.1f}% of data)')

In [ ]:
print(f'Feature per row: {X_train.shape[1]}')
print('Class balance in training set (points_finish):')
vals, counts = np.unique(y_train['points_finish'], return_counts=True)
for v, c in zip(vals, counts):
    label = 'Scored points' if v == 1 else 'No points'
    print(f'{label}: {c:,} ({c/len(y_train['points_finish'])*100:.1f}%)')
        

# Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.fit_transform(X_test)

#save scaler for 6 7 notebooks
with open('../output/models/feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

#save feature list
with open('../output/models/feature_scaler.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

print('StandardScaler fitted on training data:')
print(f'Feature means (first 5): {scaler.mean_[:5].round(3)}')
print(f'Feature stds (first 5): {scaler.scale_[:5].round(3)}')

print('Scaler saved: ../output/models/feature_scaler.pkl')
print('Feature list saved: ../output/models/feature_list.pkl')

# Helper

In [ ]:

def regression_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    if label:
        print(f'  [{label}]')
    print(f'    MAE:  {mae:.4f}')
    print(f'    RMSE: {rmse:.4f}')
    print(f'    R2:   {r2:.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}


In [ ]:

def classification_metrics(y_true, y_pred, label=''):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    if label:
        print(f'  [{label}]')
    print(f'    Accuracy:  {acc:.4f}')
    print(f'    Precision: {prec:.4f}')
    print(f'    Recall:    {rec:.4f}')
    print(f'    F1 Score:  {f1:.4f}')
    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}


In [ ]:
def save_model(model, filename):
    path = f'../output/models/{filename}'
    with open(path, 'wb') as f:
        pickle.dump(model, f)

    print(f' Saved: {path}')

In [ ]:
all_results = {}

print('Helper function defined:')
print('regression_metrics(y_true, y_pred)')
print('classification_metrics(y_true, y_pred)')
print('save_model(model, filename)')

# Section 1 Header


### Baseline Models in This Notebook

| Model | Target | Type |
|-------|--------|------|
| Linear Regression | `position` | Regression |
| Linear Regression | `points` | Regression |
| Logistic Regression | `points_finish` | Classification |

---

### What Good Looks Like for Each Target

**Position (Regression):**
- Naive baseline: always predict the mean position (~10.5) → MAE ≈ 5.5
- A good model should achieve MAE < 4.0 (within 4 positions)
- An excellent model: MAE < 3.0

**Points_finish (Classification):**
- Naive baseline: always predict majority class → accuracy ≈ 50–55%
- A good model: F1 score > 0.65
- An excellent model: F1 score > 0.75

**Points (Regression):**
- Naive baseline: always predict mean points → MAE ≈ 8–10 pts
- A good model: MAE < 6 pts
```

naive Baseline

In [ ]:
'NAIVE BASELINE (predict mean / majority class always)'

In [ ]:
#regression
for target in ['position', 'points']:
    mean_val = y_train[target].mean()
    naive_preds = np.full_like(y_val[target], fill_value=mean_val, dtype=float)
    print(f'Target: {target} (always predict mean = {mean_val:.2f})')
    metrics = regression_metrics(y_val[target], naive_preds)
    all_results[f'Naive_{target}'] = {'model': 'Naive Mean', **metrics}

In [ ]:
#classification
majority_class =int(np.bincount(y_train['points_finish'].astype(int)).argmax())
naive_clf_preds = np.full_like(y_val['points_finish'], fill_value=majority_class)
print(f'Target: points_finish (always predict {majority_class} - majority class)')
metrics= classification_metrics(y_val['points_finish'], naive_clf_preds)
all_results['Naive_points_finish'] = {'model': 'Naive Majority', **metrics}

print('These numbers are our floor - any real model must beat these')

## LINEAR REGRESSION — Target: position

In [ ]:
lr_position = LinearRegression()
lr_position.fit(X_train_scaled, y_train['position'])

val_preds_lr_pos = lr_position.predict(X_val_scaled)

val_preds_lr_pos = np.clip(val_preds_lr_pos, 1, 20)

print('Validation set performance:')
metrics = regression_metrics(y_val['position'], val_preds_lr_pos, 'Linear Regression')
all_results['LR_posotion'] = {'model': 'Linear Regression', **metrics}

In [ ]:
#show top feature coefficients
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': lr_position.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('Top 10 most influential features(by coef magnitude):')
print(coef_df.head(10).to_string(index=False))

print('Interpretation: positive coef = higher coef -> higher predicted position')
print('(higher position number = worse in F1, neg coef - better predictor)')

save_model(lr_position, 'lr_position.pkl')

## LINEAR REGRESSION — Target: points

In [ ]:

lr_points = LinearRegression()
lr_points.fit(X_train_scaled, y_train['points'])

val_preds_lr_pts = lr_points.predict(X_val_scaled)

# Points cannot be negative — clip at 0
val_preds_lr_pts = np.clip(val_preds_lr_pts, 0, 26)

print('Validation set performance:')
metrics = regression_metrics(y_val['points'], val_preds_lr_pts, 'Linear Regression')
all_results['LR_points'] = {'model': 'Linear Regression', **metrics}

print()

# Distribution of predictions vs actual
print('Prediction distribution (validation set):')
print(f'  Actual points mean:    {y_val["points"].mean():.3f}')
print(f'  Predicted points mean: {val_preds_lr_pts.mean():.3f}')
print(f'  Actual points std:     {y_val["points"].std():.3f}')
print(f'  Predicted points std:  {val_preds_lr_pts.std():.3f}')

save_model(lr_points, 'lr_points.pkl')

### LOGISTIC REGRESSION — Target: points_finish

In [ ]:
logit = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    C=1.0
)
logit.fit(X_train_scaled, y_train['points_finish'])

val_preds_logit = logit.predict(X_val_scaled)
val_proba_logit = logit.predict_proba(X_val_scaled)[:,1]

print('Validation set performance:')
metrics = classification_metrics(y_val['points_finish'], val_preds_logit,
                                 'Logistic Regression')
all_results['Logit_points_finish'] = {'model': 'Logistic Regression', **metrics}

In [ ]:
print('Detailed classification report:')
print(classification_report(
    y_val['points_finish'],
    val_preds_logit,
    target_names=['No Points', 'Scored Points']
))

logit_coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': logit.coef_[0]
}).sort_values('coefficient', ascending=False)

print('Top features INCREASING points-scoring probability (pos coef):')
print(logit_coef_df.head(5).to_string(index=False))

print('Top features DECREASING points-scoring probability(neg coef):')
print(logit_coef_df.tail(5).to_string(index=False))

save_model(logit, 'logit_points_finish.pkl')

### Baseline Summary

In [ ]:
print('Regression - Finish Position:')
print(f'Naive Mean: MAE = {all_results["Naive_position"]["MAE"]:.4f}'
      f'RMSE = {all_results["Naive_position"]["RMSE"]:.4f}'
      f"R2 = {all_results["Naive_position"]["R2"]:.4f}")

print(f'Linear Regression:   MAE = {all_results["LR_posotion"]["MAE"]:.4f}  '
      f'RMSE = {all_results["LR_posotion"]["RMSE"]:.4f}  '
      f'R² = {all_results["LR_posotion"]["R2"]:.4f}')

In [ ]:
print('Regression - Championship Points:')
print(f'Naive Mean: MAE = {all_results["Naive_points"]["MAE"]:.4f}')
print(f'Linear Regression: MAE = {all_results["LR_points"]["MAE"]:.4f}')


In [ ]:
print('Classification - Points Finish:')
print(f'Naive Majority: F1 = {all_results["Naive_points_finish"]["F1"]:.4f}')
print(f'Logistic Regression: F1 = {all_results['Logit_points_finish']['F1']:.4f}')

print('\nTarget: beat these scores with advanced models in Section 2')

# Section 2 - Advanced Models

### Three Advanced Models

| Model | Type | Key Strength |
|-------|------|-------------|
| **Random Forest** | Bagging ensemble | Robust, low variance, handles noise well |
| **Gradient Boosting** | Boosting ensemble | High accuracy, captures complex patterns |
| **XGBoost** | Optimised boosting | Fastest, often best overall performance |

---

### The Bias-Variance Trade-Off

| Model | Bias | Variance | Risk |
|-------|------|----------|------|
| Linear Regression | High | Low | Underfitting |
| Random Forest | Low | Medium | Balanced |
| Gradient Boosting | Low | Medium-High | Slight overfitting risk |
| XGBoost (tuned) | Low | Low-Medium | Best balance with tuning |


### RANDOM FOREST — Target: position

In [ ]:
rf_position = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    max_features='sqrt',
    n_jobs=-1, #use all cores
    random_state=42
)

rf_position.fit(X_train, y_train['position'])
val_preds_rf_pos = rf_position.predict(X_val)
val_preds_rf_pos = np.clip(val_preds_rf_pos, 1, 20)

print('Validation set performance:')
metrics=regression_metrics(y_val['position'], val_preds_rf_pos, 'Random Forest')
all_results['RF_position'] = {'model': 'Random Forest', **metrics}

In [ ]:
#improve over linear regression
lr_mae = all_results['LR_posotion']['MAE']
rf_mae = metrics['MAE']

print(f'Improvement over Linear Regression: '
      f'{(lr_mae - rf_mae) /lr_mae*100:+.1f}% MAE reduction')

save_model(rf_position, 'rf_position.pkl')

### RANDOM FOREST — Target: points_finish

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_clf.fit(X_train, y_train['points_finish'])
val_preds_rf_clf = rf_clf.predict(X_val)
val_proba_rf_clf = rf_clf.predict_proba(X_val)[:,1]

print('Validation set performance:')
metrics = classification_metrics(y_val['points_finish'], val_preds_rf_clf,
                                 'Random Forest')
all_results['RF_points_finish'] = {'model': 'Random Forest', **metrics}


In [ ]:
print('Detailed classification report:')
print(classification_report(
    y_val['points_finish'],
    val_preds_rf_clf,
    target_names=['No Points', 'Scored Points']
))

save_model(rf_clf, 'rf_points_finish.pkl')

### Random Forest Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,8))

for ax, model, title, target in [
    (axes[0], rf_position, 'Feature Importance\n(Position Prediction)', 'position'),
    (axes[0], rf_clf, 'Feature Importance\n(Points Finish Classification)', 'points_finish')
]:
    importances = pd.DataFrame({
        'feature':    FEATURES,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=True)

    colors = [
        '#3498DB' if f in FEATURES[:7]    else   # Driver form
        '#E67E22' if f in FEATURES[7:12]  else   # Constructor form
        '#27AE60' if f in FEATURES[12:17] else   # Track performance
        '#9B59B6' if f in FEATURES[17:]   else   # Teammate
        '#E8002D'                                 # Grid (base)
        for f in importances['feature']
    ]

    bars = ax.barh(
        importances['feature'],
        importances['importance'],
        color=colors,
        edgecolor='white',
        linewidth=0.4,
        height = 0.75
    )

    
    ax.set_xlabel('Feature Importance (Mean Impurity Reduction)', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch

legend_el = [
    Patch(facecolor='#E8002D', label='Grid (base feature)'),
    Patch(facecolor='#3498DB', label="Driver Form"),
    Patch(facecolor='#E67E22', label='Constructor Form'),
    Patch(facecolor='#9B59B6', label='Teammate Comarison'),
]

axes[1].legend(handles=legend_el, fontsize=8,
               loc='lower right', bbox_to_anchor=(1.0, 0.0))

plt.suptitle('Random Forest - Feature Importance by Target Variable',
             fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('../output/04_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('KEY INSIGHT')
top_pos  = pd.DataFrame({'f': FEATURES, 'i': rf_position.feature_importances_})\
             .sort_values('i', ascending=False).iloc[0]
top_clf  = pd.DataFrame({'f': FEATURES, 'i': rf_clf.feature_importances_})\
             .sort_values('i', ascending=False).iloc[0]
print(f'  Most important for position:      {top_pos.f} ({top_pos.i:.4f})')
print(f'  Most important for points_finish: {top_clf.f} ({top_clf.i:.4f})')


### Gradient Boosting

**Why sequential learning is powerful:**
Each tree focuses specifically on the hardest examples — the ones
previous trees got most wrong. This makes the ensemble progressively
better at difficult predictions (e.g., a backmarker driver starting P10
who scores points due to chaos ahead).

**The Bias-Variance Trade-off in Boosting:**

| Parameter | Effect on Bias | Effect on Variance |
|-----------|---------------|-------------------|
| More `n_estimators` | ↓ bias | ↑ variance (risk of overfit) |
| Lower `learning_rate` | ↓ bias (with more estimators) | ↓ variance |
| Deeper `max_depth` | ↓ bias | ↑ variance |
| More `min_samples_leaf` | ↑ bias (simpler trees) | ↓ variance |

**Tuning strategy used here:**
- Low learning rate (0.05) + many trees (500) = stable learning
- Shallow trees (max_depth=4) = low variance per tree
- This combination typically outperforms Random Forest on tabular data

**No feature scaling needed.** Gradient Boosting uses decision trees internally.


In [ ]:
gb_position = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,#small steps, stable, lower variance
    max_depth=4, #shallow trees  each corrects residuals cleanly
    min_samples_leaf=10, #each leaf needs 10+ samples - regularisation
    subsample=0.8, #stochastic gb 80% data per tree - reduces overfit
    max_features='sqrt', #random feature subset per split
    random_state=42
)


gb_position.fit(X_train, y_train['position'])

val_preds_gb_pos = gb_position.predict(X_val)
val_preds_gb_pos = np.clip(val_preds_gb_pos, 1, 20)

print('Validation set performance:')
metrics = regression_metrics(y_val['position'], val_preds_gb_pos, 'Gradient Boosting')
all_results['GB_position'] = {'model': 'Gradient Boosting', **metrics}

In [ ]:
print('Comparison vs previous models (MAE on validation):')
print(f'Naive Mean: {all_results["Naive_position"]["MAE"]:.4f}')
print(f'Linear Regression: {all_results["LR_posotion"]["MAE"]:.4f}')
print(f'Random Forest: {all_results["RF_position"]["MAE"]:.4f}')
print(f'Gradient Boosting {metrics["MAE"]:.4f} <- cur')

save_model(gb_position, 'gb_position.pkl')

## Gradient Boosting - Points Finish

In [ ]:
gb_clf = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)

gb_clf.fit(X_train, y_train['points_finish'])

val_preds_gb_clf = gb_clf.predict(X_val)
val_proba_gb_clf = gb_clf.predict_proba(X_val)[:,1]


print('Validation set performance:')
metrics = classification_metrics(y_val['points_finish'], val_preds_gb_clf,
                                 'Gradient Boosting')
all_results['GB_points_finish'] = {'model': 'Gradient Boosting', **metrics}


In [ ]:
print('Comparison vs previous classifiers (F1 on validation):')
print(f"Naive Majority: {all_results["Naive_points_finish"]["F1"]:.4f}")
print(f"Logistic Regression: {all_results["Logit_points_finish"]["F1"]:.4f}")
print(f"Random Forest: {all_results["RF_points_finish"]["F1"]:.4f}")
print(f"Gradient Boosting: {metrics["F1"]:.4f} <- current")

print('\nDetailed classification report:')
print(classification_report(
    y_val['points_finish'],
    val_preds_gb_clf,
    target_names=['No Points', 'Scored Points']
))

save_model(gb_clf, 'gb_points_finish.pkl')

## Gradient Boosting — Training Curve

In [ ]:
from sklearn.metrics import mean_absolute_error
F1_RED = "#E8002D"

train_errors = []
val_errors = []

for train_pred, val_pred in zip(
    gb_position.staged_predict(X_train),
    gb_position.staged_predict(X_val),
):
    train_errors.append(mean_absolute_error(y_train['position'], train_pred))
    val_errors.append(mean_absolute_error(y_val['position'], val_pred))

best_n = np.argmin(val_errors) +1

fig, ax = plt.subplots(figsize=(12,5))

ax.plot(range(1, len(train_errors) +1), train_errors, label='Training MAE', color='#3498DB', linewidth=1.5, alpha=0.8)
ax.plot(range(1, len(val_errors)+1), val_errors,
        label='Validation MAE', color=F1_RED, linewidth=2, alpha=0.9)


ax.axvline(best_n, color='black', linestyle='--', linewidth=1.5,
           label=f'Best n_estinators = {best_n}')
ax.scatter([best_n], [val_errors[best_n - 1]],
           color='black', s=80, zorder=5)

ax.set_xlabel('Number of Boosting Trees (n_estimators)', fontsize=12)
ax.set_ylabel('MAE - Finish Position', fontsize = 12)
ax.set_title('Gradient Boosting Training Curve - Position Prediction\n'
             '(Validation curve reveals optimal number of trees)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../output/05_gb_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## XGBoost

| Feature | Standard GB | XGBoost |
|---------|-------------|---------|
| Speed | Slower | 5–10x faster (parallelised split finding) |
| Regularisation | L2 only | L1 (alpha) + L2 (lambda) built-in |
| Missing values | Manual handling | Native NaN handling |
| Pruning | Max depth | Grows deeper then prunes (more optimal) |
| Early stopping | Not built-in | Native support (stops when val stops improving) |

**Key XGBoost-specific parameters:**

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `n_estimators` | 1000 | Upper limit (early stopping will find the real best) |
| `learning_rate` | 0.03 | Very small steps → fine-grained learning |
| `max_depth` | 5 | Slightly deeper than GB → captures more interactions |
| `subsample` | 0.8 | Row subsampling per tree |
| `colsample_bytree` | 0.8 | Feature subsampling per tree |
| `reg_alpha` | 0.1 | L1 regularisation → feature sparsity |
| `reg_lambda` | 1.0 | L2 regularisation → weight smoothing |
| `early_stopping_rounds` | 50 | Stop if val loss doesn't improve for 50 rounds |

**Early stopping** is the most important advantage:
We set n_estimators=1000 but the model automatically stops when
validation performance stops improving — preventing overfitting without
manual tuning of n_estimators.


PositionXGBOOST — Target: position

In [ ]:
if XGBOOST_AVAILABLE:

    xgb_position = xgb.XGBRegressor(
        n_estimators = 1000,
        learning_rate = 0.03,
        max_depth           = 5,
        subsample           = 0.8,
        colsample_bytree    = 0.8,
        reg_alpha           = 0.1,
        reg_lambda          = 1.0,
        objective           = 'reg:squarederror',
        eval_metric         = 'mae',
        early_stopping_rounds = 50,
        random_state        = 42,
        n_jobs              = -1,
        verbosity           = 0      # Suppress XGBoost output
    )

    xgb_position.fit(
        X_train, y_train['position'],
        eval_set=[(X_val, y_val['position'])],
        verbose=False
    )

    best_round = xgb_position.best_iteration
    val_preds_xgb_pos = xgb_position.predict(X_val)
    val_preds_xgb_pos = np.clip(val_preds_xgb_pos, 1, 20)

    print(f'Early stopping triggered at round: {best_round}\n')
    print('Validation set performance:')
    metrics = regression_metrics(y_val['position'], val_preds_xgb_pos, 'XGBoost')
    all_results['XGB_position'] = {'model': 'XGBoost', **metrics}

    print('Comparison - ALL Regression models (position):')
    for key in ['Naive_position', 'LR_posotion', 'RF_position',
                'GB_position', 'XGB_position']:
        m = all_results[key]
        print(f"{m['model']:<22} MAE={m['MAE']:.4f}"
              f"RMSE={m['RMSE']:.4f} R2={m['R2']:.4f}")
    
    save_model(xgb_position, 'xgb_position.pkl')

else:
    print('XGBoost not available. Skipping XGBoost models')
    print("Run pip install xgboost then re-run this cell")


### XGBOOST — Target: points_finish

In [ ]:
if XGBOOST_AVAILABLE:

    # Class weight for imbalanced data
    pos_weight = (y_train['points_finish'] == 0).sum() / \
                 (y_train['points_finish'] == 1).sum()
    
    xgb_clf = xgb.XGBClassifier(
        
        n_estimators          = 1000,
        learning_rate         = 0.03,
        max_depth             = 5,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        scale_pos_weight      = pos_weight,   # Handles class imbalance
        objective             = 'binary:logistic',
        eval_metric           = 'logloss',
        early_stopping_rounds = 50,
        random_state          = 42,
        n_jobs                = -1,
        verbosity             = 0
    )

    xgb_clf.fit(
        X_train, y_train['points_finish'],
        eval_set = [(X_val, y_val['points_finish'])],
        verbose=False
    )

    best_round_clf = xgb_clf.best_iteration
    val_preds_xgb_clf = xgb_clf.predict(X_val)
    val_proba_xgb_clf=xgb_clf.predict_proba(X_val)[:,1]

    print(f'Early stopping triggered at round: {best_round_clf}\n')
    print('Validation set performance')
    metrics = classification_metrics(y_val['points_finish'], val_preds_xgb_clf, 'XGBoost')
    all_results['XGB_points_finish'] = {'model': 'XGBoost', **metrics}

    print('\nComparison - ALL classifiers (points_finish):')
    for key in ['Naive_points_finish', 'Logit_points_finish', 
                'RF_points_finish', 'GB_points_finish', 'XGB_points_finish']:
        m = all_results[key]
        print(f"{m['model']:<22} F1={m['F1']:.4f}"
              f"Acc={m['Accuracy']:.4f}"
              f"Prec={m['Precision']:.4f} Rec={m['Recall']:.4f}")
        
    save_model(xgb_clf, 'xgb_points_finish.pkl')



## XGBoost Feature Importance

In [ ]:
if XGBOOST_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, model, title in [
        (axes[0], xgb_position, 'XGBoost Feature Importance\n(Position — gain)'),
        (axes[1], xgb_clf,      'XGBoost Feature Importance\n(Points Finish — gain)')
    ]:
        # ── feature_importances_ работает корректно с numpy array ──
        imp_full = pd.DataFrame({
            'feature':    FEATURES,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=True)

        colors = [
            '#3498DB' if f in FEATURES[:7]    else
            '#E67E22' if f in FEATURES[7:12]  else
            '#27AE60' if f in FEATURES[12:17] else
            '#9B59B6' if f in FEATURES[17:]   else
            '#E8002D'
            for f in imp_full['feature']
        ]

        ax.barh(
            imp_full['feature'],
            imp_full['importance'],
            color=colors,
            edgecolor='white',
            linewidth=0.4,
            height=0.75
        )

        ax.set_xlabel('Importance (F-score)', fontsize=10)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#3498DB', label='Driver Form'),
        Patch(facecolor='#E67E22', label='Constructor Form'),
        Patch(facecolor='#27AE60', label='Track Performance'),
        Patch(facecolor='#9B59B6', label='Teammate Comparison'),
        Patch(facecolor='#E8002D', label='Grid (base)'),
    ]
    axes[1].legend(handles=legend_elements, fontsize=8, loc='lower right')

    fig.suptitle('XGBoost Feature Importance',
                 fontsize=14, fontweight='bold', y=1.01)

    os.makedirs('../output', exist_ok=True)
    fig.tight_layout()
    fig.savefig('../output/05_xgb_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# Section 3 - Expected Points

### Why Expected Points Is Special

The F1 points system is non-linear and discontinuous:

| Position | Points |
|----------|--------|
| P1       | 25     |
| P2       | 18     |
| P3       | 15     |
| P4       | 12     |
| P5       | 10     |
| P6–P10   | 8, 6, 4, 2, 1 |
| P11+     | 0      |

This creates a **threshold effect**: there is a massive jump between
P10 (1 pt) and P11 (0 pts). Predicting the exact points value captures
this economic importance — finishing P11 vs P10 is more consequential
than finishing P1 vs P2 in terms of points differential.

**Expected points** is also the key input to the custom **Prediction
Reward Score** in Notebook 06.

**Model choice:** We use the best-performing model from Section 2
(typically XGBoost or Gradient Boosting) with the same configuration,
fitted on the `points` target.

**Evaluation metric:** MAE in points is most interpretable:
- MAE = 3.0 means predictions are off by ~3 championship points on average
- The winner scores 25 points — so MAE = 3 means ~12% error for race winners


## Expected Points Models - Target: points scored

In [ ]:
# Linear regression
f'Val MAE: {all_results["LR_points"]["MAE"]:.4f}'

In [ ]:
#training  random forest
rf_points = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42    
)

rf_points.fit(X_train, y_train['points'])
val_preds_rf_pts  = np.clip(rf_points.predict(X_val), 0, 26)
metrics_rf = regression_metrics(y_val['points'], val_preds_rf_pts, 'Random Forest')
all_results['RF_points'] = {'model': 'Random Forest', **metrics}
save_model(rf_points, 'rf_points.pkl')

In [ ]:
#training Gradient Boosting
gb_points = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=10,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)
gb_points.fit(X_train, y_train['points'])
val_preds_gb_pts = np.clip(gb_points.predict(X_val),0,26)
metrics_gb = regression_metrics(y_val['points'], val_preds_gb_pts, 'Gradient Boosting')
all_results['GB_points'] = {'model': 'Gradient Boosting', **metrics_gb}
save_model(gb_points, 'gb_points.pkl')

In [ ]:
#training xgboost
if XGBOOST_AVAILABLE:
    xgb_points = xgb.XGBRegressor(
        n_estimators = 1000,
        learning_rate = 0.03,
        max_depth = 5,
        subsample=0.8,
        colsample_bytree = 0.8,
        reg_alpha = 0.1,
        reg_lambda = 1.0,
        objective = 'reg:squarederror',
        eval_metric = 'mae',
        early_stopping_rounds = 50,
        random_state = 42,
        n_jobs = -1,
        verbosity = 0
    )

    xgb_points.fit(
        X_train, y_train['points'],
        eval_set = [(X_val, y_val['points'])],
        verbose = False
    )

    xgb_points.fit(
        X_train, y_train['points'],
        eval_set =[(X_val, y_val['points'])],
        verbose=False
    )

    val_preds_xgb_pts = np.clip(xgb_points.predict(X_val), 0, 26)

    metrics_xgb = regression_metrics(y_val['points'], val_preds_xgb_pts, 'XGBoost')
    all_results['XGB_points'] = {'model': 'XGBoost', **metrics_xgb}
    save_model(xgb_points, 'xgb_position.pkl')
    

## Comparison Chart

In [ ]:
pts_models = {k: v for k, v in all_results.items() if 'points' in k
              and 'finish' not in k and 'Naive' not in k}
best_pts_key = min(pts_models, key=lambda k: pts_models[k].get('MAE', 999))
best_pts_name = pts_models[best_pts_key]['model']

#get prediction from best model
if 'XGB' in best_pts_key and XGBOOST_AVAILABLE:
    best_pts_preds = val_preds_xgb_pts
elif 'GB' in best_pts_key:
    best_pts_preds = val_preds_gb_pts
elif 'RF' in best_pts_key:
    best_pts_preds = val_preds_rf_pts
else:
    best_pts_preds = val_preds_lr_pts


fig, axes = plt.subplots(1, 2, figsize=(15,6))

#left - scatter actual vs predicted
ax1 = axes[0]
ax1.scatter(y_val['points'], best_pts_preds,
            alpha=0.15, s=12, color='#3498DB', edgecolor='none')

max_val = max(y_val['points'].max(), best_pts_preds.max())
ax1.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
ax1.set_xlabel('Actual Points', fontsize=12)
ax1.set_ylabel('Predicted Points', fontsize=12)
ax1.set_title(f'Actual vs Predicted Points\n({best_pts_name})', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#right prediction error distribution
ax2 =axes[1]
errors = best_pts_preds - y_val['points']
ax2.hist(errors, bins=40, color='#3498DB', edgecolor='white',
         linewidth=0.5, alpha=0.85)
ax2.axvline(0, color = F1_RED, linewidth=2, linestyle = '--', label='Zero error')
ax2.axvline(errors.mean(), color='black', linewidth=1.5, linestyle='-',
            label=f'Mean error: {errors.mean():.2f}')
ax2.set_xlabel('Prediction Error (Predicted - Actual)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Distribution of Prediction Errors',
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle(f'Expected Points Model - {best_pts_name} - Validation Set',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../output/05_expected_points_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'Best expected points model: {best_pts_name}')
print(f'MAE: {pts_models[best_pts_key]["MAE"]:.4f} points')
print(f'RMSE: {pts_models[best_pts_key]['RMSE']:.4f} points')
print(f'R2: {pts_models[best_pts_key]["R2"]:.4f}')

# Section 4 Model Comparison & Selection

### Selection Criteria

We select the best model based on **validation set performance only.**
The test set (2025–2026) remains completely untouched until Notebook 06.

| Target | Primary Metric | Secondary Metric | Reason |
|--------|---------------|-----------------|--------|
| `position` | MAE | R² | MAE is interpretable in positions; R² shows overall fit |
| `points_finish` | F1 Score | Accuracy | F1 balances precision/recall for the binary outcome |
| `points` | MAE | RMSE | MAE is robust to the large outliers (25-pt race wins) |


## REGRESSION COMPARISON — Target: finish POSITION

In [ ]:
naive_mae_pos = all_results['Naive_position']['MAE']

reg_pos_keys = [
    ('Naive_position', '(floor)'),
    ('LR_position', ''),
    ('RF_position', ''),
    ('GB_position', ''),
]

if XGBOOST_AVAILABLE:
    reg_pos_keys.append(('XGB_position', ''))

for key, tag in reg_pos_keys:
    if key not in all_results:
        continue

    m = all_results[key]
    improvement = (naive_mae_pos - m['MAE']) / naive_mae_pos *100
    flag = ' ★' if improvement == max(
        (naive_mae_pos - all_results[k]['MAE']) / naive_mae_pos *100
        for k, _ in reg_pos_keys if k in all_results
    ) else ''
    print(f"{m['model']:<25} {m['MAE']:>10.4f} {m['RMSE']:>10.4f}"
          f"{m['R2']:>10.4f} {improvement:>+11.1f}%{flag}")

## REGRESSION COMPARISON — Target: POINTS scored

In [ ]:
# ============================================================
# COMPARISON TABLE — REGRESSION MODELS (points)
# ============================================================

naive_mae_pts = all_results['Naive_points']['MAE']

reg_pts_keys = [
    'Naive_points',
    'LR_points',
    'RF_points',
    'GB_points',
]
if XGBOOST_AVAILABLE:
    reg_pts_keys.append('XGB_points')

# ── Фильтруем: только те ключи, которые есть И имеют MAE ──
valid_pts_keys = [
    k for k in reg_pts_keys
    if k in all_results and 'MAE' in all_results[k]
]

# ── Вычисляем max improvement ДО цикла, не внутри ──────────
best_improvement = max(
    (naive_mae_pts - all_results[k]['MAE']) / naive_mae_pts * 100
    for k in valid_pts_keys
)

print('=' * 75)
print('REGRESSION COMPARISON — Target: POINTS scored')
print('=' * 75)
print(f'{"Model":<25} {"MAE":>10} {"RMSE":>10} {"R²":>10} {"vs Naive":>12}')
print('-' * 75)

for key in valid_pts_keys:
    m = all_results[key]
    improvement = (naive_mae_pts - m['MAE']) / naive_mae_pts * 100
    flag = ' ★' if abs(improvement - best_improvement) < 1e-9 else ''
    print(f"{m['model']:<25} {m['MAE']:>10.4f} {m['RMSE']:>10.4f} "
          f"{m['R2']:>10.4f} {improvement:>+11.1f}%{flag}")

print()
print('★ = best model for this target')

# ── Диагностика: какие ключи пропущены ─────────────────────
missing_keys = [k for k in reg_pts_keys if k not in valid_pts_keys]
if missing_keys:
    print()
    print(f'⚠️  Пропущены модели (не обучены или нет MAE): {missing_keys}')
    print('   Запусти соответствующие ячейки обучения и повтори.')

## CLASSIFICATION COMPARISON — Target: points_finish (0/1)

In [ ]:
clf_keys = [
    'Naive_points_finish',
    'Logit_points_finish',
    'RF_points_finish',
    'GB_points_finish',
]
if XGBOOST_AVAILABLE:
    clf_keys.append('XGB_points_finish')

best_f1 = max(all_results[k]['F1'] for k in clf_keys if k in all_results)

for key in clf_keys:
    if key not in all_results:
        continue

    m = all_results[key]
    flag = ' ★' if m['F1'] == best_f1 else ''
    print(f"{m['model']:<25} {m['F1']:>8.4f} {m['Accuracy']:>10.4f}"
          f"{m['Precision']:>11.4f} {m['Recall']:>9.4f}{flag}")
print('\n ★ = best model for this target')

## visual model comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Chart 1: Position MAE ──────────────────────────────────
ax1 = axes[0]
pos_models = ['Naive_position', 'LR_position', 'RF_position',
              'GB_position'] + (['XGB_position'] if XGBOOST_AVAILABLE else [])

# Фильтр: ключ есть И содержит MAE
valid_pos = [k for k in pos_models
             if k in all_results and 'MAE' in all_results[k]]
pos_names = [all_results[k]['model'] for k in valid_pos]
pos_maes  = [all_results[k]['MAE']   for k in valid_pos]

colors1 = [
    '#CCCCCC' if 'Naive'    in n else
    '#AED6F1' if 'Linear'   in n else
    '#2ECC71' if 'Random'   in n else
    '#F39C12' if 'Gradient' in n else
    '#E8002D'
    for n in pos_names
]
bars1 = ax1.bar(pos_names, pos_maes, color=colors1,
                edgecolor='white', linewidth=0.6, width=0.65)
for bar, val in zip(bars1, pos_maes):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.03,
             f'{val:.3f}', ha='center', va='bottom',
             fontsize=9, fontweight='bold')
ax1.set_ylabel('MAE (positions)', fontsize=11)
ax1.set_title('Finish Position\n(MAE — lower = better)',
              fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=25)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Chart 2: Points MAE ────────────────────────────────────
ax2 = axes[1]
pts_models = ['Naive_points', 'LR_points', 'RF_points',
              'GB_points'] + (['XGB_points'] if XGBOOST_AVAILABLE else [])

# Фильтр: ключ есть И содержит MAE
valid_pts = [k for k in pts_models
             if k in all_results and 'MAE' in all_results[k]]
pts_names = [all_results[k]['model'] for k in valid_pts]
pts_maes  = [all_results[k]['MAE']   for k in valid_pts]

colors2 = [
    '#CCCCCC' if 'Naive'    in n else
    '#AED6F1' if 'Linear'   in n else
    '#2ECC71' if 'Random'   in n else
    '#F39C12' if 'Gradient' in n else
    '#E8002D'
    for n in pts_names
]
bars2 = ax2.bar(pts_names, pts_maes, color=colors2,
                edgecolor='white', linewidth=0.6, width=0.65)
for bar, val in zip(bars2, pts_maes):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.05,
             f'{val:.3f}', ha='center', va='bottom',
             fontsize=9, fontweight='bold')
ax2.set_ylabel('MAE (points)', fontsize=11)
ax2.set_title('Championship Points\n(MAE — lower = better)',
              fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=25)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Chart 3: Points Finish F1 ─────────────────────────────
ax3 = axes[2]
clf_model_keys = ['Naive_points_finish', 'Logit_points_finish',
                  'RF_points_finish', 'GB_points_finish'] + \
                 (['XGB_points_finish'] if XGBOOST_AVAILABLE else [])

# Фильтр: ключ есть И содержит F1
valid_clf = [k for k in clf_model_keys
             if k in all_results and 'F1' in all_results[k]]
clf_names = [all_results[k]['model'] for k in valid_clf]
clf_f1s   = [all_results[k]['F1']   for k in valid_clf]

colors3 = [
    '#CCCCCC' if 'Naive'    in n else
    '#AED6F1' if 'Logistic' in n else
    '#2ECC71' if 'Random'   in n else
    '#F39C12' if 'Gradient' in n else
    '#E8002D'
    for n in clf_names
]
bars3 = ax3.bar(clf_names, clf_f1s, color=colors3,
                edgecolor='white', linewidth=0.6, width=0.65)
for bar, val in zip(bars3, clf_f1s):
    ax3.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.008,
             f'{val:.3f}', ha='center', va='bottom',
             fontsize=9, fontweight='bold')
ax3.set_ylabel('F1 Score', fontsize=11)
ax3.set_title('Points Finish Classification\n(F1 — higher = better)',
              fontsize=12, fontweight='bold')
ax3.set_ylim(0, 1.05)
ax3.tick_params(axis='x', rotation=25)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

fig.suptitle('All Models — Validation Set Performance Comparison',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('../output/05_model_comparison.png',   # ← outputs (с s)
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## Select Best Models

In [ ]:
def select_best(candidates, metric, higher_is_better=False):
    available = {
        k: all_results[k] 
        for k in candidates 
        if k in all_results and metric in all_results[k]
        }
    
    if not available:
        print(f"No metrics to learn {metric}")
        print(f"candidates: {candidates}")


    if higher_is_better:
        best_key = max(available, key=lambda k: available[k][metric])
    else:
        best_key = min(available, key=lambda k: available[k][metric])
    return best_key, available[best_key]

#best for position
reg_pos_candidates = ['LR_position', 'RF_position', 'GB_position',
                      'XGB_position']
best_pos_key, best_pos = select_best(reg_pos_candidates, 'MAE')

#best for points finish
clf_candidates = ['Logit_points_finish', 'RF_points_finish',
                  'GB_points_finish', 'XGB_points_finish']
best_clf_key, best_clf = select_best(clf_candidates, 'F1', higher_is_better=True)

#best for points
reg_pts_candidates = ['LR_points', 'RF_points', 'GB_points', 'XGB_points']
best_pts_key, best_pts = select_best(reg_pts_candidates, 'MAE')


In [ ]:
print(f'Target: position -> BEST: {best_pos["model"]:<22}'
      f'(MAE = {best_pos["MAE"]:.4f})')
print(f'Target: points_finish -> BEST: {best_clf["model"]:<22}'
      f'(F1 = {best_clf["F1"]:.4f})')
print(f'Target: points -> BEST: {best_pts["model"]:<22}'
      f'(MAE = {best_pts["MAE"]:.4f})')

print('\nThese models will be evaluated on the test set in Notebook 6')
print('Test set(2025-2026) has not been seen by any model during training')


# Section 5 - Save Best Models & Metadata

| File | Contents | Used In |
|------|----------|---------|
| `best_position_model.pkl` | Best regression model for position | Notebook 06, 07 |
| `best_clf_model.pkl` | Best classification model for points_finish | Notebook 06, 07 |
| `best_points_model.pkl` | Best regression model for points | Notebook 06, 07 |
| `feature_scaler.pkl` | Fitted StandardScaler (already saved) | Notebook 06, 07 |
| `feature_list.pkl` | Ordered list of feature names | Notebook 06, 07 |
| `model_metadata.pkl` | Validation metrics, model names | Notebook 06 |
| `train_test_split.pkl` | Season splits used | Notebook 06 (reproduce) |

**Why save metadata alongside models?**
When Notebook 06 loads these models, it needs to know:
- Which algorithm produced each model (for the report)
- What validation metrics were achieved (to compare with test metrics)
- Which features and scaling were used (to reconstruct input correctly)

Without metadata, a saved model is a black box. With it, Notebook 06
can automatically generate a full model card.


In [ ]:
model_map = {
    'LR_position': lr_position,
    'RF_position': rf_position,
    'GB_position': gb_position,
    'LR_points': lr_points,
    'RF_points': rf_points,
    'GB_points': gb_points,
    'Logit_points_finish': logit,
    'RF_points_finish': rf_clf,
    'GB_points_finish': gb_clf,
}
if XGBOOST_AVAILABLE:
    model_map.update({
        'XGB_position': xgb_position,
        'XGB_points': xgb_points,
        'XGB_points_finish': xgb_clf,
    })

#save best models with canonical names
best_model_object = {
    'best_position_model.pkl': model_map[best_pos_key],
    'best_clf_model.pkl': model_map[best_clf_key],
    'best_points_model.pkl': model_map[best_pts_key],
}

print('Saving best models:')
for filename, model_obj in best_model_object.items():
    save_model(model_obj, filename)

metadata = {
    'position_model': {
        'name': best_pos['model'],
        'key': best_pos_key,
        'val_mae': best_pos['MAE'],
        'val_rmse': best_pos['RMSE'],
        'val_r2': best_pos['R2'],
        'file': 'best_position_model.pkl'
    },
    'points_finish_model': {
        'name': best_pos['model'],
        'key': best_pos_key,
        'val_mae': best_pos['MAE'],
        'val_rmse': best_pos['RMSE'],
        'val_r2': best_pos['R2'],
        'file': 'best_position_model.pkl',
    },
    'points_model': {
        'name': best_pts['model'],
        'key': best_pts_key,
        'val_mae': best_pts['MAE'],
        'val_rmse': best_pts['RMSE'],
        'val_r2': best_pts['R2'],
        'file': 'best_points_model.pkl',
    },
    'features': FEATURES,
    'train_seasons': TRAIN_SEASONS,
    'val_seasons': VAL_SEASONS,
    'test_seasons': TEST_SEASONS,
    'xgboost_available': XGBOOST_AVAILABLE
}

with open('../output/models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)


split_info = {
    'train_seasons': TRAIN_SEASONS,
    'val_seasons': VAL_SEASONS,
    'test_seasons': TEST_SEASONS,
    'train_size': X_train.shape[0],
    'val_size': X_val.shape[0],
    'test_size': X_test.shape[0]
}
with open('../output/models/train_test_split.pkl', 'wb') as f:
    pickle.dump(split_info, f)

print('\nMetadata saved: ../output/models_metadata.pkl')
print('Split info saved: ../output/models/train_test_split.pkl')

## Verify Saved Files

In [ ]:
import glob

saved_files = sorted(glob.glob('../output/models/*.pkl'))

print('Verifying saved model files:\n')

all_ok = True
for path in saved_files:
    fname = path.split('/')[-1]
    try:
        with open(path, 'rb') as f:
            obj = pickle.load(f)
        size_kb = os.path.getsize(path) / 1024
        print(f'✅ {fname:<35} ({size_kb:.1f} KB)')
    except Exception as e:
        print(f'❌ {fname:<35} ERROR: {e}')
        all_ok = False

if all_ok:
    print(f'\nAll {len(saved_files)} files verified successfully')
else:
    print('WARNING: Some files failed to load. Check errors above')
    

## Full Result Table - Save to CSV

In [ ]:
results_rows = []
for key, m in all_results.items():
    row = {'experiment': key, 'model': m['model']}
    if 'MAE' in m:
        row.update({'MAE': m['MAE'], 'RMSE': m.get('RMSE'), 'R2': m.get('R2')})
    if 'F1' in m:
        row.update({'F1': m['F1'], 'Accuracy': m['Accuracy'], 
                    'Precision': m['Precision'], 'Recall': m['Recall']})
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)
results_df.to_csv('../output/05_all_model_results.csv', index=False)

print('All model results saved to: ../output/05_all_model_results.csv\n')
print(results_df.to_string(index=False))


In [ ]:
import pickle
import os

os.makedirs('../output/models', exist_ok=True)

def save_model(model, filename):
    path = f'../output/models/{filename}'
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'  Saved: {path}')

# Сохраняем лучшие модели
save_model(model_map[best_pos_key], 'best_position_model.pkl')
save_model(model_map[best_clf_key], 'best_clf_model.pkl')
save_model(model_map[best_pts_key], 'best_points_model.pkl')

# Сохраняем scaler и feature list
with open('../output/models/feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../output/models/feature_list.pkl', 'wb') as f:
    pickle.dump(FEATURES, f)

with open('../output/models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

with open('../output/models/train_test_split.pkl', 'wb') as f:
    pickle.dump(split_info, f)

print()
print('Проверка сохранённых файлов:')
for f in sorted(os.listdir('../output/models')):
    size = os.path.getsize(f'../output/models/{f}') / 1024
    print(f'  ✅  {f:<40} ({size:.1f} KB)')

In [ ]:
metadata = {
    'position_model': {
        'name':     best_pos['model'],
        'key':      best_pos_key,
        'val_mae':  best_pos['MAE'],
        'val_rmse': best_pos['RMSE'],
        'val_r2':   best_pos['R2'],
        'file':     'best_position_model.pkl'
    },
    'points_finish_model': {
        'name':     best_clf['model'],
        'key':      best_clf_key,
        'val_f1':   best_clf['F1'],        # ← именно val_f1
        'val_acc':  best_clf['Accuracy'],
        'val_prec': best_clf['Precision'],
        'val_rec':  best_clf['Recall'],
        'file':     'best_clf_model.pkl'
    },
    'points_model': {
        'name':     best_pts['model'],
        'key':      best_pts_key,
        'val_mae':  best_pts['MAE'],
        'val_rmse': best_pts['RMSE'],
        'val_r2':   best_pts['R2'],
        'file':     'best_points_model.pkl'
    },
    'features':          FEATURES,
    'train_seasons':     TRAIN_SEASONS,
    'val_seasons':       VAL_SEASONS,
    'test_seasons':      TEST_SEASONS,
    'xgboost_available': XGBOOST_AVAILABLE
}

with open('../outputs/models/model_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print('metadata пересохранён с правильными ключами.')

# Summary

## What Was Accomplished

This notebook trained **11 models** across 3 target variables,
selected the best model for each, and saved all production-ready files.

---

## Models Trained

### Baseline Models
| Model | Target | Notes |
|-------|--------|-------|
| Linear Regression | position | Requires feature scaling |
| Linear Regression | points | Requires feature scaling |
| Logistic Regression | points_finish | class_weight='balanced' |

### Advanced Models
| Model | Configuration | Key Parameter |
|-------|--------------|---------------|
| Random Forest (×3) | n_estimators=300, max_depth=8 | min_samples_leaf=10 |
| Gradient Boosting (×3) | n_estimators=500, lr=0.05 | subsample=0.8 |
| XGBoost (×3) | n_estimators=1000, lr=0.03 | early_stopping_rounds=50 |

---

## Best Models Selected

| Target | Best Model | Key Metric |
|--------|-----------|------------|
| `position` | (see results) | Lowest validation MAE |
| `points_finish` | (see results) | Highest validation F1 |
| `points` | (see results) | Lowest validation MAE |

---

## Key Engineering Decisions

**1. Temporal split (not random).**
Seasons 2018–2023 train, 2024 validates, 2025–2026 is completely held out.
Random splitting would contaminate the model with future information.

**2. Feature scaling only for linear models.**
StandardScaler fitted on training data only, then applied to val/test.
Tree-based models receive raw unscaled features — scaling is irrelevant to them.

**3. Early stopping for XGBoost.**
Setting n_estimators=1000 with early_stopping_rounds=50 finds the
optimal number of trees automatically without manual tuning.

**4. class_weight='balanced' for classifiers.**
Prevents models from exploiting class imbalance by always predicting
the majority class (no points) to achieve superficially high accuracy.

**5. Pipeline discipline.**
All models use the same train/val/test arrays. No data touches the
test set until Notebook 06. The scaler is fitted once and reused everywhere.

---

## Saved Files

| File | Description |
|------|-------------|
| `best_position_model.pkl` | Best model: finish position prediction |
| `best_clf_model.pkl` | Best model: points finish classification |
| `best_points_model.pkl` | Best model: championship points prediction |
| `feature_scaler.pkl` | Fitted StandardScaler for linear models |
| `feature_list.pkl` | Ordered list of 20 input features |
| `model_metadata.pkl` | Validation metrics + model names |
| `train_test_split.pkl` | Season split configuration |
| `05_all_model_results.csv` | Human-readable results table |
| `05_model_comparison.png` | Comparison bar chart (all models) |
| `05_gb_training_curve.png` | Gradient Boosting training curve |
| `05_xgb_feature_importance.png` | XGBoost feature importance chart |
| `05_expected_points_analysis.png` | Expected points actual vs predicted |

---

## Next Step — Notebook 06: Model Evaluation

The best models are now selected. In Notebook 06 we:
1. Evaluate all three best models on the **held-out test set (2025–2026)**
2. Compute the **custom Prediction Reward Score**
3. Analyse **where the model fails** (which drivers, circuits, conditions)
4. Generate a **full model evaluation report**
5. Decide if any model needs retraining before Notebook 07 (Driver Cards)
